# Phase 11 — Composition Bottleneck Localization and Interface Diagnosis
## NeuroForge Experimental Research Framework

### Core Research Question

Why does multi-expert execution fail to exploit the complementary computational capabilities of the existing experts strongly enough to exceed the single-expert ceiling on the validated mixed benchmark (Phase 10: k=2 = 57.2%, k=3 = 60.4%, ceiling = 66.4%)?

### Diagnostic Plan

11A (oracle feasibility) -> 11B (representation transfer) -> 11C (interface compatibility) -> 11D (aggregation) -> 11E (composition order) -> 11F (routing) -> 11G (benchmark semantics) -> 11H (minimal validated composition).

Each diagnostic is run by `neuroforge.training.phase11_composition_diagnosis.run_phase11_composition_diagnosis`. The notebook below loads the produced artifacts and derives a verdict programmatically from the data — no numbers are hardcoded.

### Mandatory Scientific Disclaimer

> Phase 10's single-expert ceiling of 66.4% is empirical, not a universal mathematical limit. Phase 11's findings are conditional on the existing expert portfolio (MLP, Graph, AttentionBlock, AttentionBlockV2) trained per Phase 6 contract.

## 1. Environment Verification
Confirm Python, PyTorch, NeuroForge import paths.

In [ ]:
import platform, sys
import torch
import neuroforge
from pathlib import Path
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

## 2. Phase 10 Baseline Reproduction
Confirm Phase 10 numbers from artifacts (Phase 10 itself is in `results/reports/phase10_multi_expert.md`).

In [ ]:
import json
phase10_path = Path('../results/metrics/phase10_multi_expert/summary.json')
if phase10_path.exists():
    with phase10_path.open('r', encoding='utf-8') as f:
        p10 = json.load(f)
    sp10 = p10.get('summary_by_policy', {})
    print('Phase 10 reproduced numbers:')
    for pol in ['k=1 Baseline (Phase 8B)', 'Fixed k=2 (Uniform C1)', 'Fixed k=3']:
        if pol in sp10:
            v = sp10[pol]
            print(f'  {pol:32s} overall={v.get("overall_accuracy_mean", 0)*100:.1f}% mixed={v.get("mixed_accuracy_mean", 0)*100:.1f}%')
else:
    print('Phase 10 summary not found; ensure results/metrics/phase10_multi_expert/summary.json exists.')

## 3. Hypothesis Registration
Pre-register the five diagnostic hypotheses BEFORE looking at results. Each is graded on real data below.

In [ ]:
HYPOTHESES = {
    'EXPERT_CAPABILITY': 'Does an oracle combination of existing experts exceed the single-expert ceiling on mixed tasks?',
    'REPRESENTATION_TRANSFER': 'Do expert intermediate states contain decodable final-target information?',
    'AGGREGATION': 'Does the choice of aggregation method (mean, weighted, concat) materially change accuracy?',
    'COMPOSITION_ORDER': 'Does the order A->B vs B->A in a sequential chain change accuracy by > 5pp?',
    'ROUTER_SELECTION': 'Does the learned router select compositions known to be useful?',
    'BENCHMARK_SEMANTICS': 'Does removing a component from the mixed benchmark flip the composite target?',
    'ROUTING_REPRESENTATION': 'Is family identity decodable from the router input (mean+std pooling)?',
    'COMPUTE_ECONOMICS': 'Does increasing k help when expert capability is limited?',
}
for k, v in HYPOTHESES.items():
    print(f'  {k}: {v}')

## 4. Run or Load Phase 11
If artifacts are missing, run the experiment via the CLI.

In [ ]:
phase11_metrics = Path('../results/metrics/phase11_composition_diagnosis')
if not (phase11_metrics / 'summary.json').exists():
    print('Phase 11 artifacts missing. Run from a terminal:')
    print('  python scripts/phase11_composition_diagnosis.py --expert-epochs 20 --samples-per-type 120')
else:
    print('Phase 11 artifacts present.')

## 5. 11A — Oracle Composition Feasibility
Without a router: does ANY combination of existing experts solve the mixed tasks better than the best single expert?

In [ ]:
summary_path = phase11_metrics / 'summary.json'
with summary_path.open('r', encoding='utf-8') as f:
    p11 = json.load(f)

oc = p11.get('oracle_ceiling_per_family', {})
p10c = p11.get('phase10_ceiling_mean', 0.0)
omixed = p11.get('oracle_ceiling_overall_mixed', 0.0)
print(f'Phase 10 single-expert ceiling (recomputed): {p10c*100:.1f}%')
print(f'Oracle (k<=3) ceiling on mixed families:    {omixed*100:.1f}%')
print()
print('Per-family oracle ceiling (best across all 10+ combinations):')
for f, v in oc.items():
    print(f'  {f}: {v*100:.1f}%')
print()
print('Best (combo) per family (across 3 seeds, majority vote):')
for f, c in p11.get('best_combos_per_family_aggregate', {}).items():
    print(f'  {f}: {c}')

## 6. 11B — Representation Transfer Probing
Per-expert, per-stage linear probe of the final target on the mixed dataset.

In [ ]:
import statistics
rep = p11.get('representation_probe_per_expert', {})
print('Decodability of FINAL target on mixed-task samples (after_blocks stage):')
for exp in ('mlp', 'graph', 'attention_v2'):
    if exp in rep:
        for stage in ('input', 'after_encoder', 'after_blocks'):
            if stage in rep[exp]:
                v = rep[exp][stage].get('final_target', 0.0)
                if isinstance(v, dict):
                    v = v.get('overall', 0.0)
                print(f'  {exp:12s} {stage:14s} -> {v*100:.1f}%')

## 7. 11C — Interface Compatibility (Sequential StateChain)

In [ ]:
iface = p11.get('interface_per_sequence', {})
mixed_families = ('FR', 'RC', 'FC', 'FRC')
print('Sequential StateChain accuracy (preserves V2 raw-feature contract):')
for seq, vals in iface.items():
    mixed = statistics.mean([vals.get('per_family_mean', {}).get(f, 0.0) for f in mixed_families])
    print(f'  {seq:35s} overall={vals.get("overall_mean", 0)*100:.1f}% mixed={mixed*100:.1f}% flops={vals.get("flops", 0):,.0f}')

## 8. 11D — Aggregation Comparison

In [ ]:
agg = p11.get('aggregation_methods', {})
print('Aggregation method comparison:')
for m, vals in agg.items():
    mixed = statistics.mean([vals.get('per_family_mean', {}).get(f, 0.0) for f in mixed_families])
    print(f'  {m:25s} overall={vals.get("overall_mean", 0)*100:.1f}% mixed={mixed*100:.1f}%')
ovs = [v.get('overall_mean', 0) for v in agg.values()]
if ovs:
    spread = max(ovs) - min(ovs)
    print(f'\nAggregation method spread (overall): {spread*100:.1f}pp')

## 9. 11E — Composition Order

In [ ]:
order = p11.get('order_per_pair', {})
print('Composition order (forward vs reverse) on mixed families:')
for pair, vals in order.items():
    fwd = statistics.mean([vals.get('forward_per_family_mean', {}).get(f, 0.0) for f in mixed_families])
    rev = statistics.mean([vals.get('reverse_per_family_mean', {}).get(f, 0.0) for f in mixed_families])
    diff = (fwd - rev) * 100
    print(f'  {pair:30s} forward={fwd*100:.1f}% reverse={rev*100:.1f}% diff={diff:+.1f}pp')

## 10. 11F — Routing Selection Diagnosis

In [ ]:
rt = p11.get('routing_selection_summary', {})
print('Router selection (Phase 10 top-2 vs oracle-useful combinations):')
for f in ('F', 'R', 'C', 'FR', 'RC', 'FC', 'FRC'):
    agr = rt.get('agreement_rate_per_family_mean', {}).get(f, 0.0)
    rec = rt.get('useful_recovery_rate_per_family_mean', {}).get(f, 0.0)
    print(f'  {f}: agreement={agr*100:.1f}% useful-recovery={rec*100:.1f}%')
print(f'\nMean k: {rt.get("mean_k_mean", 0):.2f}')
print(f'k distribution: {rt.get("k_distribution_mean", {})}')

## 11. 11G — Benchmark Semantic Validation

In [ ]:
val = p11.get('component_validation', {})
flip = val.get('component_flip_rates_mean', {})
print('Component load-bearing (target-flip rate on removal):')
for fam in ('FR', 'RC', 'FC', 'FRC'):
    d = flip.get(fam, {})
    print(f'  {fam}: F={d.get("F", 0)*100:.1f}%  R={d.get("R", 0)*100:.1f}%  C={d.get("C", 0)*100:.1f}%')

## 12. 11H — Minimal Validated Composition

In [ ]:
minc = p11.get('minimal_composition', {})
mincad = p11.get('minimal_composition_with_adapter', {})
print('Per-family accuracy using the 11A best combo (no / with linear adapter):')
for f in ('F', 'R', 'C', 'FR', 'RC', 'FC', 'FRC'):
    n = minc.get(f, 0.0)
    a = mincad.get('per_family_mean', {}).get(f, 0.0)
    print(f'  {f}: no-adapter={n*100:.1f}% with-adapter={a*100:.1f}%')
print(f'\nAdapter adds (mean): {mincad.get("mean_extra_params", 0):.0f} parameters')

## 13. Causal Diagnosis Table
Each row is decided by `build_causal_diagnosis` from the real per-seed data.

In [ ]:
causal = p11.get('causal_diagnosis_aggregate', {})
print('Causal diagnosis (status from real per-seed majority vote):')
for cat in ('EXPERT_CAPABILITY', 'REPRESENTATION_TRANSFER', 'AGGREGATION', 'COMPOSITION_ORDER', 'ROUTER_SELECTION', 'BENCHMARK_SEMANTICS', 'ROUTING_REPRESENTATION', 'COMPUTE_ECONOMICS'):
    d = causal.get(cat, {})
    print(f'\n  {cat}: {d.get("status", "INCONCLUSIVE")}')
    print(f'    Evidence: {d.get("evidence_seed_0", "")}')

## 14. Programmatic Verdict Selection (Section 36 CASE A-F)

In [ ]:
exp_cap = causal.get('EXPERT_CAPABILITY', {}).get('status', 'INCONCLUSIVE')
exp_cap_note = causal.get('EXPERT_CAPABILITY', {}).get('evidence_seed_0', '')
bench = causal.get('BENCHMARK_SEMANTICS', {}).get('status', 'INCONCLUSIVE')
rt_rep = causal.get('REPRESENTATION_TRANSFER', {}).get('status', 'INCONCLUSIVE')
agg_s = causal.get('AGGREGATION', {}).get('status', 'INCONCLUSIVE')
ord_s = causal.get('COMPOSITION_ORDER', {}).get('status', 'INCONCLUSIVE')
rt_sel = causal.get('ROUTER_SELECTION', {}).get('status', 'INCONCLUSIVE')
rt_rp = causal.get('ROUTING_REPRESENTATION', {}).get('status', 'INCONCLUSIVE')

if bench == 'NOT SUPPORTED':
    verdict = ('CASE F', 'Benchmark does not genuinely require composition')
elif exp_cap == 'NOT SUPPORTED' or 'essentially equal' in exp_cap_note or 'no multi-expert combination' in exp_cap_note:
    verdict = ('CASE E', 'No existing combination works')
elif rt_rp == 'SUPPORTED' or rt_sel == 'SUPPORTED':
    verdict = ('CASE A', 'Composition capability exists; router fails')
elif rt_rep == 'SUPPORTED':
    verdict = ('CASE B', 'Interface / representation transfer failure')
elif agg_s == 'SUPPORTED':
    verdict = ('CASE C', 'Aggregation destroys complementary information')
elif ord_s == 'SUPPORTED':
    verdict = ('CASE D', 'Order is decisive')
else:
    verdict = ('CASE G', 'Multiple interacting causes')

print(f'\nProgrammatic verdict: {verdict[0]} — {verdict[1]}')

## 15. Final Scientific Verdict

All numbers and the verdict above are derived from `results/metrics/phase11_composition_diagnosis/summary.json` (which is itself generated from per-seed real data inside `run_phase11_composition_diagnosis`).

**No numbers in this notebook are hardcoded.** Re-running the experiment (different seeds) will produce different numbers and may produce a different verdict; the same code path will be used.

The full 15-section markdown report is at `results/reports/phase11_composition_diagnosis.md`.